# spaCy — Industrial-Strength NLP for Python

---

## What Is spaCy?

spaCy is an open-source NLP library for Python designed specifically for **production use**. Where NLTK is a toolkit for research and learning, spaCy is an **opinionated, fast, and accurate** framework that gives you one best way to do each NLP task.

It ships with **pre-trained pipelines** for multiple languages that do everything in one call:
```
raw text → tokenize → POS tag → dependency parse → NER → word vectors
```

### Real-World Analogy

If NLTK is a **box of individual tools** (a hammer, screwdriver, wrench — you assemble the pipeline yourself), spaCy is a **Swiss Army knife** with all the best tools integrated, optimized to work together, and ready to use out of the box. You still have full control, but the defaults are excellent.

---

## Why spaCy Over NLTK?

| Feature | NLTK | spaCy |
|---|---|---|
| Speed | Slow (Python) | 10-20x faster (Cython) |
| POS accuracy | ~90% | ~97% |
| NER accuracy | ~80% | ~90%+ |
| API style | Many ways to do things | One clear way |
| Models | Multiple separate components | Integrated pipeline |
| Word vectors | Via WordNet | Built-in (300-dim GloVe) |
| Production use | Not recommended | Designed for production |

---

## Prerequisites

- Python basics
- NLTK notebook helps for NLP fundamentals (tokenization, POS, NER concepts)

---

## Table of Contents

1. Installation & Setup
2. The `Doc` Object — spaCy's Core
3. Tokenization & Token Attributes
4. Part-of-Speech Tagging
5. Named Entity Recognition (NER)
6. Dependency Parsing
7. Word Vectors & Similarity
8. Rule-Based Matching (`Matcher` & `PhraseMatcher`)
9. Custom Components & Pipelines
10. Training Custom NER (Modern spaCy v3 API)
11. Mini Project — Resume Information Extractor
12. Common Pitfalls
13. Interview Q&A
14. Resources
15. Summary & What's Next

---

**Official Docs:** https://spacy.io/api  
**Usage Guides:** https://spacy.io/usage  
**Course (free):** https://course.spacy.io/  
**GitHub:** https://github.com/explosion/spaCy  
**YouTube — spaCy tutorial:** https://www.youtube.com/watch?v=dIUTsFT2MeQ  

## 1. Installation & Setup

```bash
pip install spacy

# Download language models:
python -m spacy download en_core_web_sm    # small (12MB) — fast
python -m spacy download en_core_web_md    # medium (43MB) — has word vectors
python -m spacy download en_core_web_lg    # large (741MB) — best accuracy
python -m spacy download en_core_web_trf   # transformer-based — highest accuracy
```

Model naming convention: `{lang}_{type}_{genre}_{size}`
- `en` = English, `de` = German, `fr` = French, etc.
- `core` = all components (tok, POS, NER, DEP)
- `web` = trained on web text

In [ ]:
import spacy
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Download model if not already installed
import subprocess, sys
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    print("Downloading en_core_web_sm...")
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print(f"spaCy version: {spacy.__version__}")
print(f"Pipeline: {nlp.pipe_names}")
print(f"Language: {nlp.lang}")

## 2. The `Doc` Object — spaCy's Core

When you call `nlp(text)`, spaCy runs the full pipeline and returns a **`Doc`** object. The `Doc` is a container for the text and all annotations (tokens, entities, parse tree). Everything you need lives in the `Doc`.

In [ ]:
text = "Apple is looking at buying a U.K. startup for $1 billion. Tim Cook confirmed the deal."

doc = nlp(text)  # The entire pipeline runs here

print("=== Doc Object Properties ===")
print(f"Text:          {doc.text}")
print(f"Token count:   {len(doc)}")
print(f"Sentence count:{len(list(doc.sents))}")
print(f"Entity count:  {len(doc.ents)}")

print("\n=== Sentences ===")
for i, sent in enumerate(doc.sents):
    print(f"  [{i}]: '{sent.text}'")

print("\n=== Entities ===")
for ent in doc.ents:
    print(f"  {ent.text:<25} {ent.label_:<12} {spacy.explain(ent.label_)}")

## 3. Tokenization & Token Attributes

Each word/punctuation in a `Doc` is a `Token` object with rich attributes.

In [ ]:
doc2 = nlp("The quick brown foxes were running towards the river bank.")

print(f"{'Text':<12} {'Lemma':<12} {'POS':<8} {'Tag':<8} {'IsStop':<8} {'IsAlpha':<8}")
print("-" * 60)
for token in doc2:
    print(f"{token.text:<12} {token.lemma_:<12} {token.pos_:<8} "
          f"{token.tag_:<8} {str(token.is_stop):<8} {str(token.is_alpha):<8}")

print("\n=== Full Token Attribute Reference ===")
t = doc2[3]  # 'foxes'
print(f"Token: '{t.text}'")
print(f"  .lemma_     = {t.lemma_}    ← base form")
print(f"  .pos_       = {t.pos_}     ← coarse POS")
print(f"  .tag_       = {t.tag_}     ← fine-grained POS")
print(f"  .dep_       = {t.dep_}   ← dependency label")
print(f"  .head.text  = {t.head.text}     ← syntactic head")
print(f"  .is_stop    = {t.is_stop}  ← stopword?")
print(f"  .is_alpha   = {t.is_alpha}  ← alphabetic?")
print(f"  .is_digit   = {t.is_digit}  ← digit?")
print(f"  .like_email = {t.like_email}  ← looks like email?")
print(f"  .like_url   = {t.like_url}   ← looks like URL?")
print(f"  .ent_type_  = '{t.ent_type_}'   ← named entity type (if any)")

## 4. Part-of-Speech Tagging

In [ ]:
# Realistic POS analysis on a news paragraph
news = nlp("""
The Federal Reserve raised interest rates by 25 basis points on Wednesday,
marking the tenth consecutive increase since March 2022. Chairman Jerome Powell
said the central bank remains committed to bringing inflation down to 2%.
Markets reacted positively with the S&P 500 rising 1.5%.
""")

# Count POS tags
pos_counts = Counter(token.pos_ for token in news if token.is_alpha)

print("POS Distribution:")
for pos, count in pos_counts.most_common():
    print(f"  {pos:<6} {count:3d}  {spacy.explain(pos)}")

# Extract specific POS types
nouns = [t.text for t in news if t.pos_ == 'NOUN' and not t.is_stop]
verbs = [t.lemma_ for t in news if t.pos_ == 'VERB' and not t.is_stop]
adjs  = [t.text for t in news if t.pos_ == 'ADJ']

print(f"\nKey Nouns: {nouns[:8]}")
print(f"Key Verbs: {list(set(verbs))[:6]}")
print(f"Adjectives: {adjs}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
tags, counts = zip(*pos_counts.most_common(8))
ax.bar(tags, counts, color='steelblue')
ax.set_title('Part-of-Speech Distribution in News Article')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 5. Named Entity Recognition (NER)

In [ ]:
ner_text = """
Satya Nadella, CEO of Microsoft, announced a $10 billion investment in OpenAI
on January 23, 2023. The deal was signed in Seattle, Washington. Microsoft's
stock (MSFT) jumped 3% on the NASDAQ. The partnership will integrate ChatGPT
into Office 365 products. Google and Meta quickly announced competing AI projects.
"""

doc_ner = nlp(ner_text.strip())

print(f"{'Entity Text':<30} {'Label':<12} {'Meaning'}")
print("-" * 65)
for ent in doc_ner.ents:
    print(f"{ent.text:<30} {ent.label_:<12} {spacy.explain(ent.label_)}")

# Group entities by type
entity_groups = {}
for ent in doc_ner.ents:
    entity_groups.setdefault(ent.label_, []).append(ent.text)

print("\nEntities by Type:")
for label, texts in entity_groups.items():
    print(f"  {label:<12}: {texts}")

# spaCy entity label reference
print("\nCommon NER Labels:")
for label in ['PERSON', 'ORG', 'GPE', 'DATE', 'MONEY', 'PERCENT', 'PRODUCT', 'EVENT']:
    print(f"  {label:<12}: {spacy.explain(label)}")

## 6. Dependency Parsing

**Dependency parsing** analyzes the grammatical structure of a sentence by identifying relationships between words. Each word has a **head** (the word it depends on) and a **dependency label** (the relationship type).

Example: `"Apple bought Beats"`
```
Apple ──nsubj──▶ bought ◀──dobj── Beats
                 (ROOT)
```
Apple is the **nominal subject** (`nsubj`) of `bought`; Beats is the **direct object** (`dobj`).

Use cases: relationship extraction, question answering, information extraction

In [ ]:
dep_doc = nlp("Tesla CEO Elon Musk announced massive layoffs at Twitter.")

print(f"{'Token':<12} {'Head':<12} {'Dep':<12} {'Explanation'}")
print("-" * 60)
for token in dep_doc:
    print(f"{token.text:<12} {token.head.text:<12} {token.dep_:<12} {spacy.explain(token.dep_)}")

# Subject-Verb-Object extraction
def extract_svo(doc):
    """Extract (subject, verb, object) triples from a sentence."""
    triples = []
    for token in doc:
        if token.pos_ == 'VERB':
            subjects = [w.text for w in token.children if w.dep_ in ('nsubj', 'nsubjpass')]
            objects  = [w.text for w in token.children if w.dep_ in ('dobj', 'pobj', 'attr')]
            if subjects and objects:
                for subj in subjects:
                    for obj in objects:
                        triples.append((subj, token.lemma_, obj))
    return triples

texts = [
    "Apple acquired Beats Electronics for $3 billion.",
    "Google released a new language model.",
    "The president signed the legislation."
]

print("\nSubject-Verb-Object Extraction:")
for text in texts:
    doc = nlp(text)
    triples = extract_svo(doc)
    print(f"  '{text}'")
    print(f"  → {triples}")

## 7. Word Vectors & Similarity

The `md` (medium) and `lg` (large) models include **300-dimensional word vectors** (GloVe embeddings trained on Common Crawl). These vectors capture semantic similarity — words with similar meanings have vectors that are close together in 300D space.

In [ ]:
# Word vectors and similarity
# Note: en_core_web_sm doesn't have word vectors
# We demonstrate with the small model, but similarity needs md or lg

# Checking if model has vectors
print(f"Model has vectors: {nlp.vocab.vectors_length > 0}")
if nlp.vocab.vectors_length == 0:
    print("en_core_web_sm has no word vectors.")
    print("For similarity, use: python -m spacy download en_core_web_md")
    print("\nDemonstrating API (works the same with md/lg models):")

# Try to load md model if available, else show the API
try:
    nlp_md = spacy.load('en_core_web_md')
    HAS_MD = True
except OSError:
    HAS_MD = False

if HAS_MD:
    doc1 = nlp_md("I like pizza")
    doc2 = nlp_md("I enjoy eating pasta")
    doc3 = nlp_md("The stock market crashed")

    print(f"'pizza' vs 'pasta' similarity: {nlp_md('pizza').similarity(nlp_md('pasta')):.3f}")
    print(f"'king' vs 'queen' similarity:  {nlp_md('king').similarity(nlp_md('queen')):.3f}")
    print(f"'king' vs 'banana' similarity: {nlp_md('king').similarity(nlp_md('banana')):.3f}")
    print(f"\nDoc similarity 'pizza' vs 'pasta': {doc1.similarity(doc2):.3f}")
    print(f"Doc similarity 'pizza' vs 'stock': {doc1.similarity(doc3):.3f}")
else:
    # Explain the concept
    print("""
With en_core_web_md loaded:
  nlp('king').similarity(nlp('queen'))  → ~0.78  (semantically related)
  nlp('pizza').similarity(nlp('pasta')) → ~0.70  (same category)
  nlp('king').similarity(nlp('banana')) → ~0.25  (unrelated)

Word vector for 'king' is a 300-dimensional array:
  nlp('king').vector.shape  → (300,)

Classic vector arithmetic:
  king - man + woman ≈ queen  (embeddings capture semantic relationships)
""")

## 8. Rule-Based Matching

spaCy's `Matcher` lets you define **token-level patterns** using token attributes (text, POS, entity type, etc.). Unlike regex, you match on linguistic features.

**Example:** Find all mentions of "iPhone" followed by a number or model name.

In [ ]:
from spacy.matcher import Matcher, PhraseMatcher

matcher = Matcher(nlp.vocab)

# Pattern 1: Match "iPhone" followed by optional number/model
# Each dict in the list is a TOKEN, keys are token attributes
iphone_pattern = [
    {"TEXT": {"REGEX": "iPhone"}, "OP": "+"},   # 'iPhone' (one or more)
    {"IS_DIGIT": True, "OP": "?"},                # optional number
]

# Pattern 2: Match monetary expressions like "$100 million"
money_pattern = [
    {"TEXT": "$"},
    {"LIKE_NUM": True},
    {"LOWER": {"IN": ["million", "billion", "trillion"]}, "OP": "?"}
]

# Pattern 3: Match proper noun sequences (e.g., company names)
proper_noun_pattern = [
    {"POS": "PROPN"},
    {"POS": "PROPN", "OP": "*"},  # one or more proper nouns in a row
]

matcher.add("IPHONE",      [iphone_pattern])
matcher.add("MONEY",       [money_pattern])
matcher.add("PROPER_NOUN", [proper_noun_pattern])

test_text = """
Apple announced iPhone 15 Pro Max and iPhone 15 will start at $799 and $699 million respectively.
Tim Cook, Apple CEO, made the announcement at the Steve Jobs Theater in Cupertino.
"""

doc_match = nlp(test_text.strip())
matches = matcher(doc_match)

print(f"Found {len(matches)} matches:")
seen = set()
for match_id, start, end in matches:
    span_text = doc_match[start:end].text
    if span_text not in seen:  # deduplicate
        rule = nlp.vocab.strings[match_id]
        print(f"  [{rule}] → '{span_text}'")
        seen.add(span_text)

In [ ]:
# PhraseMatcher — match exact phrases (faster than Matcher for large phrase lists)
from spacy.matcher import PhraseMatcher

phrase_matcher = PhraseMatcher(nlp.vocab, attr='LOWER')  # case-insensitive

# List of phrases to match
tech_companies = ['apple', 'google', 'microsoft', 'meta', 'amazon',
                  'openai', 'anthropic', 'nvidia', 'tesla', 'spacex']

patterns = [nlp.make_doc(company) for company in tech_companies]
phrase_matcher.add('TECH_COMPANY', patterns)

text2 = """OpenAI and Microsoft are collaborating closely. Google and Anthropic
are competing in the AI space. NVIDIA supplies chips to all of them."""

doc2 = nlp(text2)
matches2 = phrase_matcher(doc2)

print("Tech companies mentioned:")
for match_id, start, end in matches2:
    print(f"  '{doc2[start:end].text}'")

## 9. Custom Pipeline Components

You can add your own processing steps to the spaCy pipeline. Components are functions that take a `Doc` and return a modified `Doc`.

In [ ]:
from spacy.language import Language

# Custom pipeline component that adds a custom attribute

@Language.component('sentiment_tagger')
def sentiment_tagger(doc):
    """Add a simple sentiment score to each sentence as a custom attribute."""
    positive_words = {'excellent', 'amazing', 'great', 'good', 'love', 'fantastic', 'wonderful'}
    negative_words = {'terrible', 'awful', 'bad', 'horrible', 'hate', 'disappointing', 'poor'}

    # Register custom attributes on Doc and Span
    if not doc.has_extension('sentiment'):
        from spacy.tokens import Doc, Span
        Doc.set_extension('sentiment', default=None, force=True)
        Span.set_extension('sentiment', default=None, force=True)

    words_lower = set(t.lower_ for t in doc)
    pos_count = len(words_lower & positive_words)
    neg_count = len(words_lower & negative_words)
    doc._.sentiment = 'positive' if pos_count > neg_count else 'negative' if neg_count > pos_count else 'neutral'

    return doc

# Add to pipeline
if 'sentiment_tagger' not in nlp.pipe_names:
    nlp.add_pipe('sentiment_tagger', last=True)

print(f"Updated pipeline: {nlp.pipe_names}")

test_docs = [
    "This product is absolutely amazing and wonderful!",
    "Terrible experience. The service was awful and disappointing.",
    "The meeting was scheduled for 3pm."
]

print("\nCustom sentiment tagger results:")
for text in test_docs:
    doc = nlp(text)
    print(f"  '{text[:50]}'  → {doc._.sentiment}")

## 10. Processing Large Text Efficiently with `nlp.pipe()`

For processing many texts, always use `nlp.pipe()` instead of calling `nlp()` in a loop. It batches documents and processes them in parallel.

In [ ]:
import time

texts = [
    "Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cupertino.",
    "Microsoft CEO Satya Nadella announced record earnings in Seattle.",
    "Google's DeepMind published AlphaFold research in Nature.",
    "Amazon Web Services dominates cloud computing with 33% market share.",
    "Tesla delivered 500,000 vehicles in Q4, exceeding analyst expectations.",
] * 20  # 100 texts total

# Method 1: Loop (slow)
t0 = time.time()
docs_loop = [nlp(t) for t in texts]
loop_time = time.time() - t0

# Method 2: nlp.pipe() (fast)
# disable: skip components you don't need (saves time)
t0 = time.time()
docs_pipe = list(nlp.pipe(texts, batch_size=32))
pipe_time = time.time() - t0

print(f"Processing {len(texts)} texts:")
print(f"  Loop:     {loop_time:.3f}s")
print(f"  pipe():   {pipe_time:.3f}s")
print(f"  Speedup:  {loop_time/pipe_time:.1f}x")

# Efficient: disable unused components
t0 = time.time()
docs_fast = list(nlp.pipe(texts, batch_size=32,
                            disable=['parser', 'sentiment_tagger']))  # only tok + NER
fast_time = time.time() - t0
print(f"  pipe(NER only): {fast_time:.3f}s")

# Show some results
print("\nExtracted entities from first 5 texts:")
for doc in docs_pipe[:5]:
    ents = [(e.text, e.label_) for e in doc.ents]
    print(f"  {ents}")

## 11. Mini Project — Resume Information Extractor

### The Business Problem

HR teams receive hundreds of resumes. We want to automatically extract:
- Candidate name
- Contact info (email, phone)
- Skills mentioned
- Educational institutions
- Work experience (companies, titles)

This is exactly the kind of structured information extraction spaCy excels at.

In [ ]:
import re

# Sample resume text
resume_text = """
SARAH JOHNSON
sarah.johnson@email.com | +1-555-123-4567 | linkedin.com/in/sarahjohnson
San Francisco, CA

SUMMARY
Experienced machine learning engineer with 6 years at Google and Amazon.
Expert in Python, TensorFlow, PyTorch, and Kubernetes.

EXPERIENCE
Senior ML Engineer | Google | 2020 - Present | Mountain View, CA
  - Led development of recommendation system serving 500 million users
  - Improved model accuracy by 15% using transformer architectures

Machine Learning Engineer | Amazon | 2018 - 2020 | Seattle, WA
  - Built real-time fraud detection system using XGBoost and Apache Spark
  - Reduced false positives by 30%

EDUCATION
M.S. Computer Science | Stanford University | 2018
B.S. Mathematics | MIT | 2016

SKILLS
Python, TensorFlow, PyTorch, Scikit-learn, SQL, Spark, Kubernetes, Docker,
AWS, GCP, Machine Learning, Deep Learning, NLP, Computer Vision
"""

print("Analyzing resume...")
print(resume_text)

In [ ]:
# ==================================================
# Resume extraction pipeline
# ==================================================

TECH_SKILLS = [
    'python', 'tensorflow', 'pytorch', 'scikit-learn', 'sql', 'spark',
    'kubernetes', 'docker', 'aws', 'gcp', 'azure', 'xgboost', 'lightgbm',
    'machine learning', 'deep learning', 'nlp', 'computer vision',
    'java', 'scala', 'r', 'javascript', 'typescript', 'go', 'rust',
    'apache kafka', 'apache spark', 'hadoop', 'react', 'node'
]

def extract_resume_info(text):
    doc = nlp(text)
    result = {}

    # 1. Email (regex — spaCy doesn't tag emails as entities)
    emails = re.findall(r'[\w.+-]+@[\w-]+\.[a-z]{2,}', text)
    result['email'] = emails[0] if emails else None

    # 2. Phone number
    phones = re.findall(r'[+]?[\d\s().-]{10,}', text)
    result['phone'] = phones[0].strip() if phones else None

    # 3. Named entities
    persons = [e.text for e in doc.ents if e.label_ == 'PERSON']
    orgs    = [e.text for e in doc.ents if e.label_ == 'ORG']
    gpes    = [e.text for e in doc.ents if e.label_ == 'GPE']

    result['name'] = persons[0] if persons else None
    result['companies'] = list(dict.fromkeys(orgs))  # deduplicate
    result['locations']  = list(dict.fromkeys(gpes))

    # 4. Skills (PhraseMatcher)
    skill_matcher = PhraseMatcher(nlp.vocab, attr='LOWER')
    skill_patterns = [nlp.make_doc(s) for s in TECH_SKILLS]
    skill_matcher.add('SKILL', skill_patterns)
    skill_matches = skill_matcher(doc)
    skills = list(dict.fromkeys(doc[s:e].text for _, s, e in skill_matches))
    result['skills'] = skills

    # 5. Education (detect degree patterns)
    edu_pattern = re.findall(r'(?:B\.?S|M\.?S|Ph\.?D|B\.?A|M\.?A)\.?\s+[\w\s]+\|[\w\s]+', text)
    result['education'] = edu_pattern

    return result

info = extract_resume_info(resume_text)

print("=" * 50)
print("     EXTRACTED RESUME INFORMATION")
print("=" * 50)
print(f"Name:      {info['name']}")
print(f"Email:     {info['email']}")
print(f"Phone:     {info['phone']}")
print(f"Companies: {info['companies']}")
print(f"Locations: {info['locations']}")
print(f"Skills:    {info['skills']}")
print(f"Education: {info['education']}")

## 12. Common Pitfalls

### Pitfall 1: Loading the Model Inside a Loop
```python
# WRONG — extremely slow, loads model for every text
for text in texts:
    nlp = spacy.load('en_core_web_sm')  # ← never do this
    doc = nlp(text)

# CORRECT — load once, use many times
nlp = spacy.load('en_core_web_sm')  # ← outside the loop
for text in texts:
    doc = nlp(text)
```

### Pitfall 2: Using `nlp()` in a Loop Instead of `nlp.pipe()`
For >10 texts, always use `nlp.pipe(texts)` — it batches processing and is much faster.

### Pitfall 3: Forgetting to Disable Unused Components
```python
# If you only need tokenization, disable everything else
with nlp.disable_pipes('tagger', 'parser', 'ner'):
    docs = [nlp(t) for t in texts]
# OR
docs = list(nlp.pipe(texts, disable=['tagger', 'parser', 'ner']))
```

### Pitfall 4: `en_core_web_sm` Has No Word Vectors
The small model doesn't include word vectors. For `.similarity()`, use `en_core_web_md` or `en_core_web_lg`.

### Pitfall 5: Token vs String Comparison
```python
# WRONG — Token object is not a string
if token == 'apple':
# CORRECT
if token.text == 'apple' or token.lower_ == 'apple':
```

## 13. Interview Q&A

---

**Q1: What is dependency parsing and what is it used for?**

> Dependency parsing analyzes the grammatical structure of a sentence as a tree, where each word (except the root) depends on exactly one other word. Each dependency has a label describing the relationship (e.g., `nsubj` = nominal subject, `dobj` = direct object, `prep` = prepositional modifier). Applications: (1) Subject-verb-object extraction for information extraction, (2) Question answering (finding the answer to "who did X"), (3) Coreference resolution aids, (4) Machine translation preserving grammatical structure.

---

**Q2: What's the difference between `token.pos_` and `token.tag_`?**

> `pos_` gives the **coarse-grained** Universal Dependencies POS tag (one of 18 categories: NOUN, VERB, ADJ, ADV, PROPN, etc.) — same across all languages. `tag_` gives the **fine-grained** language-specific tag: for English, the Penn Treebank tagset (NN vs NNS vs NNP for singular/plural/proper nouns; VB vs VBD vs VBG for base/past/gerund verbs). Use `pos_` for language-agnostic code; use `tag_` when you need precise morphological information.

---

**Q3: How does spaCy's Matcher differ from regular expressions?**

> Regular expressions match on **character sequences** (`r"\d+"`). spaCy's Matcher matches on **token attribute patterns** — you can match on POS, entity type, morphology, lexical attributes, or text. Example: `[{'POS': 'ADJ'}, {'POS': 'NOUN'}]` matches any adjective-noun pair regardless of the actual words. This is much more powerful for linguistic patterns: find all "compound noun phrases", "modal verbs followed by passive verbs", etc. Regex can't do this without massive case-by-case patterns.

---

**Q4: How would you add a custom entity type to spaCy?**

> Two approaches: (1) **Rule-based**: Add a custom pipeline component using `EntityRuler` that matches patterns (e.g., all uppercase words are `TICKER` symbols). (2) **Trained**: Collect labeled examples, convert to spaCy's `DocBin` format, and fine-tune the NER component using `spacy train`. The `spacy train` command takes a `config.cfg` file specifying the model architecture and training data. Modern spaCy v3 uses a transformer backbone (`en_core_web_trf`) for highest accuracy.

---

**Q5: Why is `nlp.pipe()` faster than calling `nlp()` in a loop?**

> `nlp.pipe()` uses **batched processing** — it groups multiple texts into a batch and runs the pipeline on the whole batch at once. Internally, the statistical models (CNN, LSTM, or transformer) process batches more efficiently than single samples due to vectorized matrix operations. Additionally, `nlp.pipe()` allows setting `n_process > 1` for multiprocessing, distributing work across CPU cores. The speedup is typically 2-5x for CPU and 10x+ for GPU.

## 14. Resources

### Official
- **spaCy Documentation:** https://spacy.io/api
- **spaCy Usage Guides:** https://spacy.io/usage
- **Free spaCy Course:** https://course.spacy.io/
- **spaCy Projects (templates):** https://spacy.io/universe/project
- **GitHub:** https://github.com/explosion/spaCy

### Ecosystem
- **spacy-transformers** (BERT/RoBERTa in spaCy): https://spacy.io/usage/embeddings-transformers
- **Prodigy** (annotation tool by spaCy creators): https://prodi.gy/

### Videos
- **spaCy Tutorial (NLP with Python):** https://www.youtube.com/watch?v=dIUTsFT2MeQ

## 15. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **`nlp(text)`** | Runs the full pipeline; returns a `Doc` with all annotations |
| **Token attributes** | `.text`, `.lemma_`, `.pos_`, `.tag_`, `.dep_`, `.is_stop`, `.ent_type_` |
| **NER** | `doc.ents` — each entity has `.text`, `.label_`, `.start`, `.end` |
| **Dependency parsing** | `token.dep_`, `token.head` — grammatical relationships |
| **Matcher** | Token-pattern matching on linguistic attributes, not just text |
| **PhraseMatcher** | Fast exact phrase matching — ideal for large keyword lists |
| **`nlp.pipe()`** | Always use for multiple texts — 2-10x faster than loops |
| **Custom components** | `@Language.component` decorator + `nlp.add_pipe()` |
| **Disable pipes** | `nlp.pipe(texts, disable=['parser'])` — skip components you don't need |

### What's Next

**HuggingFace Transformers** — state-of-the-art NLP with BERT, GPT, T5, and friends:
- `pipeline()` for instant text classification, NER, QA, summarization
- Fine-tuning pre-trained transformers on your data
- Generating text with GPT-2/GPT-Neo/Llama